# Fracture characterisation: HTI and AVAz

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_fracture_avaz_hti.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 8.4 Building a Fractured Model
2. 8.2 Hudson and Linear Slip
3. 8.5 Azimuthal AVO (AVAz)
4. 8.6 Shear-Wave Splitting

In [ ]:
# Ask Ogbon starter: Characterise a fractured reservoir
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""A fractured reservoir: one set of dry, aligned vertical cracks in the cemented sand of Part 8.

Follows Sections 8.1 (Cracks to Stiffness), 8.2 (Hudson and Linear Slip), 7.3
(The Thomsen Parameters), 8.5 (Azimuthal AVO, Rueger 1998) and 8.6 (Shear-Wave
Splitting). The cracks are dry; Section 8.3 says what filling them changes.
Units: moduli GPa, density g/cc, so velocities from them are km/s (the square
root of GPa over g/cc); the shale is in m/s; a thickness in m over km/s is ms.
Assumptions and limits:
- Dilute, aligned penny cracks added to the host's compliance (linear slip).
- The swing is Rueger's formula: an approximation, first order in the contrasts
  and the anisotropy and best below about 30 degrees; Section 8.5 shows it can
  miss even the sign of the swing, and the figure prints the exact coefficient.
- Splitting is a vertical shear wave through a uniform layer: no tuning, no
  attenuation, the delay alone.
- The field: the cemented sand is the field earth's reservoir; its cracks strike
  with the faults and are densest beside them. Rueger's swing is read along the
  reservoir, and a noisy azimuthal survey at the most fractured trace is fitted
  with cos 2(azimuth) to find the strike.

It prints the numbers Figure 12.5 computes for the same answers as name = value
lines; then it builds the field earth every starter shares, prints what it measures
there as [field] name = value lines, and draws one figure. Python 3.9+, numpy and
matplotlib; the earth's random numbers are seeded, so every run prints the same.
Edit the parameters and run again.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
E_CRACK = 0.03             # crack density of the aligned set (dimensionless); the cracks are dry
K_HOST, G_HOST, RHO_HOST = 10.397110881545897, 12.430435125431675, 2.12  # the cemented sand, every crack shut (GPa, GPa, g/cc)
THICKNESS_M = 1000.0       # thickness of the fractured interval (m)
THETA_DEG = 30.0           # incidence angle for the azimuthal swing (degrees)
INCLUDE_SPLITTING = False  # True with narrow azimuths: read the splitting delay, not the AVAz swing

In [ ]:
# --- the book's constants (not interview answers) ---
# the book's standard rocks (SSM.STANDARD, Section 3.3): P velocity (m/s), S velocity (m/s), density (g/cc)
ROCKS = {
    'shale': (2800, 1241.3793103448277, 2.2521031547808517),
    'brine sand': (2889.973997793982, 1504.4719169684909, 2.2545487499999997),
    'oil sand': (2543.1993596875604, 1529.5017037311773, 2.1813626422963988),
    'gas sand': (2341.92404704112, 1582.4208998550723, 2.0379042775100316)
}
AZ_STEP_DEG = 15                # azimuths 0 to 180 degrees from the fracture normal
STRIKE_DEG = 30.0               # the field's faults, and the cracks with them, strike 30 degrees east of north
DAMAGE_M = 200.0                # the damage zone: the crack density falls by e over this distance from a fault (m)
NOISE = 0.1                     # the AVAz measurement's noise, as a share of the swing

def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)

def validate(e_crack, theta_deg, thickness_m):
    """Stop with a clear message on a parameter the model cannot use."""
    if not 0 <= e_crack < 0.1:
        raise ValueError('E_CRACK = %g is out of range: the crack model is for dilute cracks, 0 to 0.1' % e_crack)
    if not 0 <= theta_deg < 90 or thickness_m < 0 or min(K_HOST, G_HOST, RHO_HOST) <= 0:
        raise ValueError('need 0 <= THETA_DEG < 90, THICKNESS_M >= 0 and a positive host')

def cracked_stiffness(k0, g0, e):
    """Voigt stiffness (GPa) of the host plus one dry crack set, the normal along axis 1 (Section 8.2)."""
    # the host's compliance plus the cracks' (Schoenberg and Sayers 1995), inverted
    E = 9 * k0 * g0 / (3 * k0 + g0)                # Young's modulus
    pr = (3 * k0 - 2 * g0) / (2 * (3 * k0 + g0))   # Poisson's ratio
    bn = 16 * (1 - pr ** 2) / (3 * E)       # normal compliance of the cracks per unit density
    bt = bn / (1 - pr / 2)                  # tangential compliance
    S = np.zeros((6, 6))
    S[:3, :3] = -pr / E
    S[0, 0] = S[1, 1] = S[2, 2] = 1 / E
    S[3, 3] = S[4, 4] = S[5, 5] = 2 * (1 + pr) / E
    S[0, 0] += e * bn
    S[4, 4] += e * bt
    S[5, 5] += e * bt
    return np.linalg.inv(S)

def ruger_rpp(top, bot, theta_deg, az_deg):
    """Rueger's (1998) P-P coefficient, isotropic top over HTI bottom, azimuth from the symmetry axis (Section 8.5)."""
    # top is (vp, vs, density); bot adds eps_v, delta_v and Rueger's gamma (positive for cracks)
    vp = (top[0] + bot[0]) / 2; vs = (top[1] + bot[1]) / 2
    z1, z2 = top[2] * top[0], bot[2] * bot[0]
    g1, g2 = top[2] * top[1] ** 2, bot[2] * bot[1] ** 2
    k2 = (2 * vs / vp) ** 2
    eps_b, delta_b, gamma_b = bot[3], bot[4], bot[5]
    A = 0.5 * (z2 - z1) / ((z1 + z2) / 2)
    dvp = (bot[0] - top[0]) / vp
    b_iso = 0.5 * (dvp - k2 * (g2 - g1) / ((g1 + g2) / 2))
    b_ani = 0.5 * (delta_b + 2 * k2 * gamma_b)
    c2 = np.cos(np.radians(az_deg)) ** 2
    c_iso = 0.5 * dvp
    c_ani = 0.5 * (eps_b * c2 * c2 + delta_b * (1 - c2) * c2)
    th = np.radians(theta_deg)
    s2 = np.sin(th) ** 2
    return A + (b_iso + b_ani * c2) * s2 + (c_iso + c_ani) * s2 * np.tan(th) ** 2

def hti_sand(e):
    """The cracked sand as Rueger's bottom medium: vp, fast vs (m/s), density, eps_v, delta_v, Rueger's gamma."""
    C = cracked_stiffness(K_HOST, G_HOST, e)
    c_along, c_vert, c13, c44, c55 = C[0, 0], C[2, 2], C[0, 2], C[3, 3], C[4, 4]   # axis 1 the normal, 3 vertical
    eps_v = (c_along - c_vert) / (2 * c_vert)
    delta_v = ((c13 + c55) ** 2 - (c_vert - c55) ** 2) / (2 * c_vert * (c_vert - c55))
    return (1e3 * np.sqrt(c_vert / RHO_HOST), 1e3 * np.sqrt(c44 / RHO_HOST), RHO_HOST, eps_v, delta_v, (c44 - c55) / (2 * c55))

def plot_field(e, dens, xr, swing, az, amp, fit, strike):
    """The cracks in the section, the swing along the reservoir, and the azimuthal AVO at the most fractured trace."""
    fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.3), gridspec_kw={'width_ratios': [1.6, 1, 1]})
    ext = [0, e['x'][-1] / 1e3, e['z'][-1], 0]
    ax[0].imshow(e['vp'] / 1e3, cmap='Greys', vmin=0.0, vmax=1.6 * e['vp'].max() / 1e3, aspect='auto', extent=ext)
    im = ax[0].imshow(np.where(e['res'] & (dens > 0), dens, np.nan), cmap='plasma', aspect='auto', extent=ext)
    fz, fx = np.nonzero(e['fault'])
    ax[0].plot(e['x'][fx] / 1e3, e['z'][fz], 'c.', ms=1)
    fig.colorbar(im, ax=ax[0], label='crack density')
    style(ax[0], 'The reservoir\'s cracks, densest beside the faults (cyan)', 'distance (km)', 'depth (m)', grid=False)
    ax[1].plot(xr / 1e3, swing, 'k.', ms=3)
    style(ax[1], 'Swing at %.0f degrees along the reservoir' % THETA_DEG, 'distance (km)', 'across minus along')
    ax[2].plot(az, amp, 'o', ms=4, label='measured, with noise')
    ax[2].plot(np.arange(181), fit, 'k', lw=1, label='fitted cos 2(azimuth)')
    ax[2].axvline(strike, color='r', ls='--', label='strike found, %.1f degrees' % strike)
    style(ax[2], 'AVAz at the most fractured trace', 'acquisition azimuth (degrees from north)', 'P-P coefficient')
    ax[2].legend(fontsize=7)
    fig.tight_layout()
    plt.show()

def main():
    validate(E_CRACK, THETA_DEG, THICKNESS_M)
    C = cracked_stiffness(K_HOST, G_HOST, E_CRACK)
    c_along, c_vert, c13, c44, c55, c66 = C[0, 0], C[2, 2], C[0, 2], C[3, 3], C[4, 4], C[5, 5]   # axis 1 the normal, 3 vertical
    # Thomsen's parameters in the vertical frame (Section 7.3)
    eps_v = (c_along - c_vert) / (2 * c_vert)
    delta_v = ((c13 + c55) ** 2 - (c_vert - c55) ** 2) / (2 * c_vert * (c_vert - c55))
    gamma_r = (c44 - c55) / (2 * c55)          # Rueger's gamma, positive for cracks
    gamma_v = -gamma_r / (1 + 2 * gamma_r)     # Tsvankin's gamma(V), minus half the tangential weakness
    delta_t = 1 - c55 / c44                    # the tangential weakness (Section 8.2)
    report('eps_v', eps_v)
    report('delta_v', delta_v)
    report('gamma_v', gamma_v)
    report('delta_t', delta_t)
    # the sand with the cracks under the standard shale
    vp_host, vs_host = np.sqrt(c_vert / RHO_HOST), np.sqrt(c44 / RHO_HOST)   # km/s: vertical P, fast vertical S
    bot = (vp_host * 1000.0, vs_host * 1000.0, RHO_HOST, eps_v, delta_v, gamma_r)
    az_deg = np.arange(0, 181, AZ_STEP_DEG)
    amp = np.array([ruger_rpp(ROCKS['shale'], bot, THETA_DEG, p) for p in az_deg])
    along = int(90 // AZ_STEP_DEG)                 # the index of 90 degrees, along the fractures
    report('rueger_across', amp[0])
    report('rueger_along', amp[along])
    report('rueger_swing', amp[0] - amp[along])
    # splitting: the fast wave keeps the host's shear modulus, the slow one has c55
    t_fast = THICKNESS_M / vs_host                 # m over km/s: ms
    t_slow = THICKNESS_M / np.sqrt(c55 / RHO_HOST)
    report('splitting_delay_ms', t_slow - t_fast, '%.4f')
    # self-checks: dry aligned linear-slip cracks give Thomsen's epsilon 8 e / 3 in the crack frame, whatever the host
    assert abs(eps_v + (8 * E_CRACK / 3) / (1 + 16 * E_CRACK / 3)) < 1e-9
    assert t_slow >= t_fast and 0 <= delta_t < 1
    # the field: the cemented sand is the reservoir; its cracks strike with the faults, densest in their damage zone
    host = hti_sand(0.0)[:3]
    e = field_earth(seal=ROCKS['shale'], upper=host, lower=host)
    fz, fx = np.nonzero(e['fault'])
    cols = np.nonzero(e['res'].any(axis=0) & (e['ntg'] == 1))[0]
    iz = np.argmax(e['res'][:, cols], axis=0)                       # the top of the reservoir, trace by trace
    d = np.sqrt((e['x'][cols, None] - e['x'][fx]) ** 2 + (e['z'][iz, None] - e['z'][fz]) ** 2).min(axis=1)
    dens_r = np.minimum(E_CRACK * (0.5 + np.exp(-d / DAMAGE_M)), 0.099)   # half E_CRACK far away, 1.5 times beside a fault
    swing = np.array([ruger_rpp((e['vp'][k - 1, c], e['vs'][k - 1, c], e['den'][k - 1, c]), hti_sand(q), THETA_DEG, 0.0)
                      - ruger_rpp((e['vp'][k - 1, c], e['vs'][k - 1, c], e['den'][k - 1, c]), hti_sand(q), THETA_DEG, 90.0)
                      for c, k, q in zip(cols, iz, dens_r)])
    j = int(np.argmax(dens_r))
    c, k = cols[j], iz[j]
    above = (e['vp'][k - 1, c], e['vs'][k - 1, c], e['den'][k - 1, c])
    az = np.arange(0, 180, AZ_STEP_DEG)                             # the survey's azimuths, from north
    rng = np.random.default_rng(5)
    meas = np.array([ruger_rpp(above, hti_sand(dens_r[j]), THETA_DEG, p - STRIKE_DEG - 90.0) for p in az])
    meas = meas + NOISE * abs(swing[j]) * rng.standard_normal(az.size)
    G = np.column_stack([np.ones(az.size), np.cos(2 * np.radians(az)), np.sin(2 * np.radians(az))])
    a0, a1, a2 = np.linalg.lstsq(G, meas, rcond=None)[0]
    peak = 0.5 * np.degrees(np.arctan2(a2, a1)) % 180.0              # the azimuth of the largest coefficient
    strike = (peak - 90.0) % 180.0 if swing[j] > 0 else peak         # across the cracks is larger when the swing is positive
    fieldnote('reservoir_traces', cols.size, '%d')
    fieldnote('crack_density_max', dens_r.max(), '%.4f')
    fieldnote('swing_max', swing.max(), '%.5f')
    fieldnote('swing_min', swing.min(), '%.5f')
    fieldnote('strike_true_deg', STRIKE_DEG, '%.1f')
    fieldnote('strike_found_deg', strike, '%.1f')
    dens = np.zeros(e['vp'].shape)
    dens[:, cols] = dens_r
    fit = a0 + a1 * np.cos(2 * np.radians(np.arange(181))) + a2 * np.sin(2 * np.radians(np.arange(181)))
    plot_field(e, dens, e['x'][cols], swing, az, meas, fit, strike)

if __name__ == '__main__':
    main()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)